### Import packages

In [2]:
import os
from openai import OpenAI
import pandas as pd
import json
import tqdm

### Check if environment variable is set for OpenAI API Key

In [3]:
try:
    api_key = os.environ["OPENAI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Environment variable OPENAI_API_KEY missing.") from exc

### Define which OpenAI model to use

In [4]:
MODEL = 'gpt-5-nano'

### Construct OpenAI client instance

In [5]:
client = OpenAI(api_key=api_key)

def send_LLM(prompt: str) -> str:
    try:
        # Send chat request
        chat_completion = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a rigorous JSON-only annotator of online sarcasm."},
                {"role": "user", "content": prompt}
            ])
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    return chat_completion.choices[0].message.content

### Define schema for sarcasm as JSON

In [6]:
SARCASM_SCHEMA = {
    "sarcasm": {
        "sarcasm_score": "1-5 integer (1 = not sarcastic, 5 = very sarcastic)",
        "confidence_score": "1-5 integer (1 = not confident, 5 = very confident)",
        "justification": "string - explanation based on observed language",
        "sarcasm_markers": ["string"]
    } 
}

def prompt_sarcasm_analysis(thread_title: str, comment: str) -> str:
    schema_str = json.dumps(SARCASM_SCHEMA, indent=2)
    return f"""
You are a rigorous JSON-only annotator of online sarcasm.
Analyze the COMMENT under the THREAD TITLE below.

Return ONLY one JSON object that strictly matches this schema:

{schema_str}

THREAD TITLE:
\"\"\"{thread_title}\"\"\"

COMMENT:
\"\"\"{comment}\"\"\"

"""

### Read input data

In [7]:
df = pd.read_csv('data.csv')

df.head()

,id,thread_title,comment
0,1,Mehrheit sieht ältere im Vorteil – nicht einma...,Solange alle ihre Tiefkühlpizza und RTL haben ...
1,2,Mehrheit sieht ältere im Vorteil – nicht einma...,Ich bin schockiert.
2,3,Mehrheit sieht ältere im Vorteil – nicht einma...,"Würden sich die babyboomer zu Tode arbeiten, b..."
3,4,Mehrheit sieht ältere im Vorteil – nicht einma...,Auf den Schreck erstmal eine Rentenerhöhung.
4,5,Mehrheit sieht ältere im Vorteil – nicht einma...,Finde den Generationen-Vertrag an sich schon d...


### Perform LLM sarcasm analysis on every comment and save as JSON

In [8]:
import json

results = []

for _, row in df.iterrows():
    row_id = row["id"]
    thread_title = row["thread_title"]
    comment = row["comment"]

    prompt = prompt_sarcasm_analysis(thread_title, comment)
    llm_answer = send_LLM(prompt)

    # Verify correct output format
    try:
        parsed = json.loads(llm_answer)
        sarcasm_obj = parsed.get("sarcasm", {})
    except json.JSONDecodeError:
        sarcasm_obj = {"raw_response": llm_answer}

    results.append({
        "id": row_id,
        "thread_title": thread_title,
        "comment": comment,
        "sarcasm": sarcasm_obj
    })

# Save concatenated results as JSON array
with open("sarcasm_results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

# Show in notebook
results[:2]

[{'id': 1,
  'thread_title': 'Mehrheit sieht ältere im Vorteil – nicht einmal ein Drittel der jungen glaubt an faire Rente',
  'comment': 'Solange alle ihre Tiefkühlpizza und RTL haben ist doch alles super!',
  'sarcasm': {'sarcasm_score': 4,
   'confidence_score': 4,
   'justification': "The comment uses ironic praise ('alles super') to downplay a serious issue about pension fairness. It implies that trivial comforts (frozen pizza and RTL) make the situation tolerable, which signals sarcasm through mockery and downplaying the broader problem.",
   'sarcasm_markers': ['irony',
    'mocking tone',
    'downplaying serious issue via trivial pleasures']}},
 {'id': 2,
  'thread_title': 'Mehrheit sieht ältere im Vorteil – nicht einmal ein Drittel der jungen glaubt an faire Rente',
  'comment': 'Ich bin schockiert.',
  'sarcasm': {'sarcasm_score': '1',
   'confidence_score': '5',
   'justification': "The comment 'Ich bin schockiert.' is a straightforward expression of emotion (shock) without